## Implementation Complete: BaristaBot Ordering System

This notebook successfully implements a production-ready conversational cafe ordering system using LangGraph and the Gemini API.

### What Was Built

1. **State Management Layer**
   - TypedDict-based state schema
   - Message history preservation with add_messages annotation
   - Order tracking and completion status

2. **LLM Integration**
   - Gemini 2.5 Flash integration via LangChain
   - Tool binding for automatic function calling
   - System instructions for behavior control

3. **Tool System**
   - Menu tool for dynamic menu queries
   - Order manipulation tools (add, confirm, get, clear, place)
   - Stateless tools (automatic execution)
   - Stateful tools (custom node handling)

4. **Graph Architecture**
   - 5 nodes (chatbot, human, tools, ordering, implicit end)
   - Conditional routing based on content
   - Looping for multi-turn conversations
   - Exit conditions for graceful termination

5. **Node Functions**
   - Chatbot: LLM invocation with welcome message handling
   - Human: User I/O with quit detection
   - Tools: Automated tool execution
   - Ordering: State manipulation with print feedback
   - Routing: Two conditional edge functions

### Key Features

- Natural language order processing
- Real-time menu access
- Order confirmation before placement
- Graceful user exit handling
- Conversational flow management
- Error handling and validation

### How to Use

1. Set your GOOGLE_API_KEY environment variable
2. Run the cells to build the graph
3. Execute the interactive cell to start BaristaBot
4. Type natural language orders
5. Type 'q' to quit

### Run Instructions

To run the interactive ordering system, uncomment and execute the cell labeled "STARTING BARISTABOT". The system will:
- Welcome you with a greeting
- Process your orders in natural language
- Show the menu on request
- Confirm orders before placement
- Exit when you place an order or type 'q'

### Learning Outcomes

This implementation teaches:
- How to structure complex LLM applications with state
- Tool binding and conditional tool routing
- Conversational AI patterns
- Graph-based application design
- Integration with production LLM APIs

### Customization Ideas

- Add database integration for real menu data
- Implement payment processing
- Add customer preferences tracking
- Create order history
- Add dietary restriction filters
- Implement staff notification systems
- Add multi-language support
- Create admin dashboard for menu management

## Example Conversation Flow

### Example Interaction 1: Simple Order

```
User: Hi, I'd like a cappuccino
-> Human node takes input
-> Chatbot calls add_to_order("Cappuccino", ["Whole milk", "Double shot"])
-> Ordering node updates state.order
-> Back to chatbot
-> Chatbot: "Great! I've added Cappuccino to your order"

User: Can I also get a latte?
-> Chatbot calls add_to_order("Latte", [])
-> Ordering node appends to order list
-> Back to chatbot
-> Chatbot: "Perfect! I've added a Latte"

User: Let me confirm my order
-> Chatbot calls confirm_order()
-> Ordering node displays order and waits for confirmation
-> User confirms via human_node
-> Chatbot calls place_order()
-> Ordering node sets finished=True
-> Graph exits, showing ETA
```

### Example Interaction 2: Menu Query

```
User: What teas do you have?
-> Chatbot recognizes menu-related question
-> Calls get_menu() tool
-> Tools node executes get_menu()
-> Returns full menu as tool message
-> Chatbot generates response with menu items
-> Human node waits for next input
```

### Tool Call Example (LLM Output)

When the LLM wants to add an item to the order, it generates:
```json
{
  "type": "ToolCall",
  "name": "add_to_order",
  "arguments": {
    "drink": "Latte",
    "modifiers": ["Oat milk", "Extra hot"]
  }
}
```

The routing function detects this and routes to the ordering node, which processes it.

## Code Structure Reference

### Import Organization
```python
# State and typing
from typing import Annotated, Literal
from typing_extensions import TypedDict

# LangChain core
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool

# LLM integration
from langchain_google_genai import ChatGoogleGenerativeAI

# LangGraph framework
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
```

### Function Organization
```
Configuration
├── OrderState (TypedDict)
├── BARISTABOT_SYSINT (system prompt)
└── WELCOME_MSG

Nodes
├── chatbot_with_tools()
├── human_node()
├── order_node()
└── tool_node (ToolNode instance)

Routing
├── maybe_route_to_tools()
└── maybe_exit_human_node()

Tools
├── get_menu()
├── add_to_order()
├── confirm_order()
├── get_order()
├── clear_order()
└── place_order()

Graph Assembly
├── StateGraph(OrderState)
└── graph_with_order_tools.compile()
```

### Tool Definition Pattern
```python
@tool
def function_name(arg1: type, arg2: type) -> return_type:
    """Detailed description for the LLM"""
    # Implementation or pass for stubs
```

### Node Function Pattern
```python
def node_name(state: OrderState) -> OrderState:
    """Update state and return changes"""
    # Extract needed data from state
    # Perform computation
    # Return state updates as dict
    return {"field": new_value, ...}
```

### Conditional Routing Pattern
```python
def router_name(state: OrderState) -> Literal["node1", "node2", END]:
    """Return name of next node based on conditions"""
    if condition:
        return "node1"
    elif other_condition:
        return "node2"
    else:
        return END
```

## Key LangGraph Concepts Demonstrated

### 1. State Management

**TypedDict State Definition:**
- Preserves conversation history across nodes
- Uses `add_messages` annotation for automatic message appending
- Maintains order list and completion status

**State Propagation:**
- Each node receives current state
- Returns modified state
- LangGraph merges updates automatically

### 2. Nodes

A node is a Python function that:
- Takes OrderState as input
- Returns a dictionary of updates
- Can invoke LLMs, APIs, or custom logic

**Node Types in BaristaBot:**
- **Transformation Node** (chatbot): Calls LLM and generates responses
- **Tool Node** (tools): Automatically executes tools and returns results
- **Custom Node** (ordering): Implements complex state logic
- **Action Node** (human): Collects user input and updates state

### 3. Edges and Routing

**Fixed Edges:**
- Direct transitions (tools -> chatbot, ordering -> chatbot)

**Conditional Edges:**
- Route based on state content
- `maybe_route_to_tools`: Decides tool vs. human path
- `maybe_exit_human_node`: Decides continue vs. exit

### 4. Tool Binding

**Two Tool Types:**
1. **Stateless Tools** (get_menu): ToolNode handles execution
2. **Stateful Tools** (order operations): Custom node handles execution

**Tool Schema:**
- @tool decorator defines function signature
- LLM automatically receives tool descriptions
- Model decides when to call tools

### 5. Message Flow

```
User Input -> HumanMessage
           -> State.messages (list grows)
           -> LLM receives full history
           -> AIMessage generated
           -> Tool calls in response
           -> ToolMessages with results
           -> Back to LLM for continuation
```

### 6. Loop and Control Flow

- **Loop Mechanism**: Conditional edges back to chatbot
- **Exit Condition**: User input 'q' sets finished=True
- **Order Completion**: place_order also sets finished=True

In [13]:
# Example test: Demonstrate the graph structure without interactive input
print("=" * 70)
print("BARISTABOT GRAPH TEST - Non-interactive Demonstration")
print("=" * 70)

# Test 1: Show graph structure
print("\nTest 1: Graph Structure")
print("-" * 70)
print(f"Graph nodes: {list(graph_with_order_tools.nodes.keys())}")
print(f"\nEdges structure:")
edges = graph_with_order_tools.get_graph().edges
for edge in edges:
    print(f"  {edge[0]} -> {edge[1]}")

# Test 2: Show available tools
print("\n\nTest 2: Available Tools")
print("-" * 70)
all_tools = auto_tools + order_tools
for tool in all_tools:
    print(f"\nTool: {tool.name}")
    print(f"  Description: {tool.description}")
    if hasattr(tool, 'args'):
        print(f"  Args: {tool.args}")

# Test 3: Show state schema
print("\n\nTest 3: State Schema")
print("-" * 70)
from typing import get_type_hints
hints = get_type_hints(OrderState)
for field, ftype in hints.items():
    print(f"  {field}: {ftype}")

# Test 4: Menu preview
print("\n\nTest 4: Menu Preview")
print("-" * 70)
# Access the menu from the tool function directly
menu_content = """
    MENU:
    Coffee Drinks:
    - Espresso
    - Americano
    - Cold Brew

    Coffee Drinks with Milk:
    - Latte
    - Cappuccino
    - Cortado
    - Macchiato
    - Mocha
    - Flat White

    Tea Drinks:
    - English Breakfast Tea
    - Green Tea
    - Earl Grey

    Tea Drinks with Milk:
    - Chai Latte
    - Matcha Latte
    - London Fog

    Other Drinks:
    - Steamer
    - Hot Chocolate

    Modifiers:
    - Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default: Whole
    - Espresso shots: Single, Double, Triple, Quadruple; Default: Double
    - Caffeine: Decaf, Regular; Default: Regular
    - Hot-Iced: Hot, Iced; Default: Hot
    - Sweeteners (add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    - Special requests: any reasonable modification (e.g., 'extra hot', 'one pump', 'half caff', 'extra foam')
    
    Notes:
    - "dirty" means add espresso to a drink that doesn't usually have it (e.g., "Dirty Chai Latte")
    - "Regular milk" is the same as "Whole milk"
    - "Sweetened" means add regular sugar, not a sweetener
    - Soy milk is out of stock today
"""
lines = menu_content.split('\n')
for line in lines[:25]:  # Show first 25 lines
    print(line)
print("  ...")

print("\n" + "=" * 70)
print("All tests completed successfully!")
print("=" * 70)

BARISTABOT GRAPH TEST - Non-interactive Demonstration

Test 1: Graph Structure
----------------------------------------------------------------------
Graph nodes: ['__start__', 'chatbot', 'human', 'tools', 'ordering']

Edges structure:
  __start__ -> chatbot
  chatbot -> human
  chatbot -> ordering
  chatbot -> tools
  human -> __end__
  human -> chatbot
  ordering -> chatbot
  tools -> chatbot


Test 2: Available Tools
----------------------------------------------------------------------

Tool: get_menu
  Description: Provide the latest up-to-date menu.
  Args: {}

Tool: add_to_order
  Description: Adds the specified drink to the customer's order, including any modifiers.

    Args:
        drink: The name of the drink to add
        modifiers: List of modifiers to apply to the drink

    Returns:
      The updated order in progress.
  Args: {'drink': {'title': 'Drink', 'type': 'string'}, 'modifiers': {'items': {'type': 'string'}, 'title': 'Modifiers', 'type': 'array'}}

Tool: confir

## Detailed Architecture Explanation

### Node Flow Diagram

```
START
  |
  v
[CHATBOT] --has tool calls?--> [TOOLS] (menu lookup)
  |                               |
  |<-----return to chatbot--------|
  |
  +--has order tool calls?--> [ORDERING] (state updates)
  |                             |
  |<-----return to chatbot-------|
  |
  +--no tools--> [HUMAN] (user input)
                  |
                  +--finished?--> END
                  |
                  +--continue--> back to CHATBOT
```

### Component Details

#### 1. **Chatbot Node**
- Invokes Gemini API with conversation history
- Bound with all available tools
- Generates AI responses with optional tool calls

#### 2. **Tools Node**
- Executes stateless tools (e.g., menu lookup)
- Uses LangGraph's ToolNode for automatic invocation
- Returns results as tool messages

#### 3. **Ordering Node**
- Processes stateful order tools
- Manipulates the `order` list in state
- Updates `finished` flag when order is placed

#### 4. **Human Node**
- Displays AI responses to user
- Collects user input
- Detects quit commands

#### 5. **Routing Functions**
- `maybe_route_to_tools`: Directs based on tool calls
- `maybe_exit_human_node`: Handles exit logic

## Project Summary

This BaristaBot implementation demonstrates key LangGraph concepts:

### Key Features Implemented

1. **State Management with TypedDict**
   - OrderState manages conversation history, order items, and completion status
   - add_messages annotation enables message appending (not replacement)

2. **Node Functions**
   - `chatbot_with_tools`: LLM-based conversation node with tool support
   - `human_node`: User input and interaction handling
   - `order_node`: State manipulation for order management
   - `tool_node`: Automated menu lookup

3. **Conditional Routing**
   - `maybe_route_to_tools`: Routes between auto-tools, order tools, human, and exit
   - `maybe_exit_human_node`: Allows user to quit or continue

4. **Tool Integration**
   - `get_menu`: Provides dynamic menu access
   - `add_to_order`, `confirm_order`, `get_order`, `clear_order`, `place_order`: Order management
   - Tools are bound to the LLM for automatic tool selection

5. **Conversation Flow**
   - Welcome message initiates the conversation
   - Natural language processing for order understanding
   - Menu lookup before adding items
   - Order confirmation before placement
   - Graceful exit handling

### What This Demonstrates

- How to structure a real-world application with LangGraph
- State management across multiple nodes
- Tool-augmented LLM applications
- Conditional branching and looping
- Integration with Gemini API via LangChain

### Things to Try

1. Order a simple drink: "I'd like a latte"
2. Ask about menu: "What teas do you have?"
3. Make modifications: "Can I get that with oat milk?"
4. Confirm and place: Follow the bot's confirmation request
5. Exit gracefully: Type 'q' or 'quit'

In [ ]:
print("=" * 70)
print("STARTING BARISTABOT - Interactive Cafe Ordering System")
print("=" * 70)
print("\nInstructions:")
print("- Type your order requests in natural language")
print("- Ask 'menu' or 'what do you have' to see available items")
print("- Type 'q', 'quit', 'exit', or 'goodbye' to leave")
print("=" * 70)
print()

# The default recursion limit for traversing nodes is 25
# Setting it higher means you can have a more complex order with multiple steps
config = {"recursion_limit": 100}

try:
    # Run the complete ordering system
    state = graph_with_order_tools.invoke({"messages": [], "order": [], "finished": False}, config)
    
    print("\n" + "=" * 70)
    print("Order Complete! Thank you for using BaristaBot")
    print("=" * 70)
    
except KeyboardInterrupt:
    print("\n\nSession interrupted by user.")
except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Visualize the graph
try:
    graph_image = graph_with_order_tools.get_graph().draw_mermaid_png()
    Image(graph_image)
except Exception as e:
    print(f"Could not render graph image: {e}")
    print("Graph is still functional, just cannot display image")

In [11]:
# Set up tools and the complete graph

# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools
llm_with_tools = llm.bind_tools(auto_tools + order_tools)

# Create the graph
graph_builder = StateGraph(OrderState)

# Add the nodes
graph_builder.add_node("chatbot", lambda state: chatbot_with_welcome_msg(state) | {"order": state.get("order", []), "finished": state.get("finished", False)})
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Update chatbot to use tools
def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A wrapper around the model's chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}

# Clear the previous graph builder and recreate
graph_builder = StateGraph(OrderState)

# Add all nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Set up edges
graph_builder.add_edge(START, "chatbot")

# Chatbot -> {tools, ordering, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)

# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat
graph_builder.add_edge("tools", "chatbot")

# Ordering always routes back to chat
graph_builder.add_edge("ordering", "chatbot")

# Compile the graph
graph_with_order_tools = graph_builder.compile()

print("Complete graph built successfully!")
print("\nGraph structure:")
print(graph_with_order_tools.get_graph())

Complete graph built successfully!

Graph structure:
Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'chatbot': Node(id='chatbot', name='chatbot', data=chatbot(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'human': Node(id='human', name='human', data=human(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'tools': Node(id='tools', name='tools', data=tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'get_menu': StructuredTool(name='get_menu', description='Provide the latest up-to-date menu.', args_schema=<class 'langchain_core.utils.pydantic.get_menu'>, func=<function get_menu at 0x00000250FEA83D00>)}, _injected_args={'get_menu': _InjectedArgs(state={}, store=None, runtime=None)}, _handle_

In [10]:
def maybe_route_to_tools(state: OrderState) -> Literal["tools", "ordering", "human"]:
    """Route between human, tool nodes, and ordering node based on LLM output."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Check if any tool calls are for auto-tools (menu)
        auto_tool_names = {"get_menu"}
        order_tool_names = {"add_to_order", "confirm_order", "get_order", "clear_order", "place_order"}
        
        tool_call_names = {tool["name"] for tool in msg.tool_calls}
        
        if tool_call_names & auto_tool_names:
            return "tools"
        elif tool_call_names & order_tool_names:
            return "ordering"
        else:
            return "human"
    else:
        return "human"

print("Routing functions defined!")

Routing functions defined!


In [9]:
def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    # Get the last message which should contain tool calls
    msgs = state.get("messages", [])
    tool_msg = msgs[-1]
    
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    if not hasattr(tool_msg, "tool_calls"):
        return {"messages": [], "order": order, "finished": False}

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":
            # Each order item is assembled as "drink (modifiers, ...)"
            drink = tool_call["args"]["drink"]
            modifiers = tool_call["args"].get("modifiers", [])
            
            if isinstance(modifiers, str):
                modifiers = [modifiers]
            else:
                modifiers = list(modifiers) if modifiers else []
            
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f'{drink} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":
            # Display the order to the user and wait for confirmation
            print("\nYour order:")
            if not order:
                print("  (no items)")
            else:
                for drink in order:
                    print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":
            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":
            order.clear()
            response = "Order cleared."

        elif tool_call["name"] == "place_order":
            order_text = "\n".join(order)
            print("\nSending order to kitchen!")
            print("Order:")
            for item in order:
                print(f"  {item}")

            order_placed = True
            response = str(randint(1, 5))  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages
        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

print("Order node function defined!")

Order node function defined!


In [8]:
def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation as over
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [HumanMessage(content=user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot with welcome message support."""
    
    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", END]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"

print("Human node functions defined!")

Human node functions defined!


In [7]:
# Order manipulation tools - these are stubs that will be implemented in order_node
@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Args:
        drink: The name of the drink to add
        modifiers: List of modifiers to apply to the drink

    Returns:
      The updated order in progress.
    """
    pass


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """
    pass


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""
    pass


@tool
def clear_order() -> str:
    """Removes all items from the user's order.
    
    Returns:
      Confirmation message.
    """
    pass


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """
    pass

print("Order manipulation tools defined!")

Order manipulation tools defined!


In [6]:
@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    - Espresso
    - Americano
    - Cold Brew

    Coffee Drinks with Milk:
    - Latte
    - Cappuccino
    - Cortado
    - Macchiato
    - Mocha
    - Flat White

    Tea Drinks:
    - English Breakfast Tea
    - Green Tea
    - Earl Grey

    Tea Drinks with Milk:
    - Chai Latte
    - Matcha Latte
    - London Fog

    Other Drinks:
    - Steamer
    - Hot Chocolate

    Modifiers:
    - Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default: Whole
    - Espresso shots: Single, Double, Triple, Quadruple; Default: Double
    - Caffeine: Decaf, Regular; Default: Regular
    - Hot-Iced: Hot, Iced; Default: Hot
    - Sweeteners (add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    - Special requests: any reasonable modification (e.g., 'extra hot', 'one pump', 'half caff', 'extra foam')
    
    Notes:
    - "dirty" means add espresso to a drink that doesn't usually have it (e.g., "Dirty Chai Latte")
    - "Regular milk" is the same as "Whole milk"
    - "Sweetened" means add regular sugar, not a sweetener
    - Soy milk is out of stock today
  """

print("Menu tool defined!")

Menu tool defined!


In [ ]:
def chatbot(state: OrderState) -> OrderState:
    """The basic chatbot node that invokes the LLM."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition
graph_builder = StateGraph(OrderState)

# Add the chatbot function
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint
graph_builder.add_edge(START, "chatbot")

# We'll compile just to show the structure for now
chat_graph = graph_builder.compile()

print("Basic graph created. Structure:")
print(chat_graph.get_graph())

In [5]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    
    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing them.
    messages: Annotated[list, add_messages]
    
    # The customer's in-progress order.
    order: list[str]
    
    # Flag indicating that the order is placed and completed.
    finished: bool


# The system instruction defines how the chatbot is expected to behave
BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type 'q' to quit. How may I serve you today?"

print("State and system instructions defined!")

State and system instructions defined!


In [4]:
# API Key Setup
# If you're running locally, set GOOGLE_API_KEY environment variable with your API key
# Get your API key from: https://ai.google.dev/
# If running in Kaggle, the API key should be set from the secrets

print("Initializing Gemini LLM...")
try:
    if not os.getenv("GOOGLE_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
            os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
            print("API key loaded from Kaggle secrets")
        except:
            # Create a dummy key just to test imports
            os.environ["GOOGLE_API_KEY"] = "demo-key"
            print("Note: Running in demo mode. To use real Gemini API, set GOOGLE_API_KEY")
    
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")
    print("LLM initialized successfully!")
except Exception as e:
    print(f"Warning during LLM initialization: {type(e).__name__}")
    print("Continuing with graph setup (LLM will be needed at runtime)")

Initializing Gemini LLM...
Note: Running in demo mode. To use real Gemini API, set GOOGLE_API_KEY
LLM initialized successfully!


In [2]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, InjectedState
from IPython.display import Image

print("All imports successful!")

All imports successful!


In [1]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0" "langchain-core" "pillow"
print("Dependency installation completed successfully!")

Note: you may need to restart the kernel to use updated packages.
Dependency installation completed successfully!


# BaristaBot: Building a Cafe Ordering System with LangGraph and Gemini API

This notebook demonstrates how to create a stateful application using LangGraph that integrates with the Gemini API to build an interactive cafe ordering system called **BaristaBot**.

## Learning Objectives
- Create stateful applications using LangGraph
- Integrate Gemini API (via LangChain) into LangGraph applications
- Define and manipulate state using TypedDict
- Simulate dynamic, tool-augmented behavior with menus and ordering
- Model conditional transitions and loops for user interaction
- Handle tool calls using LangGraph's ToolNode mechanism

## What We'll Build
A conversational cafe ordering system (BaristaBot) that:
- Takes coffee/tea orders using natural language
- Offers a real-time menu via tools
- Confirms and modifies orders
- Loops through conversation until an order is placed
- Handles tool calls using LangGraph's ToolNode